
# 🌳 Agrupamiento Jerárquico con SciPy (Dendrograma) y scikit-learn (AgglomerativeClustering)

**Autor:** Tú 🙂  

**Fecha de generación:** 2025-08-26 19:09

---

Este cuaderno implementa paso a paso el **agrupamiento jerárquico aglomerativo**, integrando:

1) **SciPy** para construir el **dendrograma** a partir de la **matriz de distancias**.  
2) **scikit-learn** para entrenar el modelo `AgglomerativeClustering` y asignar una etiqueta de **cluster** a cada muestra.

**Dataset:** `Iris` (scikit-learn). Usamos dos variables para facilitar la visualización 2D:
- `sepal length (cm)`
- `sepal width (cm)`

---

## 🎯 Objetivos de Aprendizaje
- Comprender el **proceso progresivo** del clustering jerárquico (**aglomerativo** vs **divisivo**).
- Construir y **visualizar** un **dendrograma** con `SciPy`.
- **Elegir k** (número de clusters) a partir del dendrograma (intuición y ejemplo).
- Aplicar `AgglomerativeClustering` (scikit-learn) y **visualizar clusters** en 2D.
- Contrastar etiquetas generadas por `SciPy` y `scikit-learn`.

> **Nota:** Este notebook prioriza el **modo maestro de ciencia de datos**: teoría → código → interpretación.



## 🧠 Teoría mínima necesaria

### ¿Qué es el clustering jerárquico?
Técnica de **aprendizaje no supervisado** que organiza observaciones en una **jerarquía** de grupos (**clusters**). Produce un **árbol** (dendrograma) donde cada fusión representa la unión de dos clusters.

### Modos de construcción
- **Aglomerativo (bottom-up):** parte con cada muestra como un cluster y fusiona progresivamente los dos clusters **más similares** hasta formar uno solo. Es el enfoque más usado.
- **Divisivo (top-down):** parte de un único cluster con todas las muestras y lo **divide** recursivamente hasta clusters unitarios. Es más costoso y menos común.

### Distancias y *linkage*
Para decidir qué clusters fusionar se define:
- **Métrica de distancia** (euclídea en este cuaderno).
- **Criterio de enlace (linkage):**
  - `single`: mínima distancia entre pares de puntos de dos clusters (puede formar cadenas).
  - `complete`: máxima distancia entre pares (clusters compactos).
  - `average`: promedio de distancias entre todos los pares.
  - `ward`: minimiza la **varianza intra-cluster** en cada fusión (común con distancia euclídea).

### Dendrograma e interpretación
- Eje **Y**: nivel de **distancia/disimilitud** en que se produce cada fusión.
- Cortar el dendrograma a una **altura** (umbral de distancia) define el número de clusters.
- **Estrategia práctica:** buscar **brechas** verticales grandes para elegir un corte razonable.

### Escalamiento
Antes de calcular distancias, **estandarizar** características suele mejorar resultados (evita que una variable domine por escala). Usaremos `StandardScaler`.



## ⚙️ Requisitos
- Python 3.8+
- `numpy`, `matplotlib`, `scikit-learn`, `scipy`

Puedes instalar/actualizar con:
```bash
pip install -U numpy matplotlib scikit-learn scipy
```


In [ ]:

# Imports y versiones
from __future__ import annotations

import numpy as np
import matplotlib.pyplot as plt

from sklearn import datasets
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import AgglomerativeClustering

from scipy.spatial.distance import pdist, squareform
from scipy.cluster.hierarchy import linkage, dendrogram, fcluster

import sys, sklearn, scipy
print(f"Python: {sys.version.split()[0]} | numpy: {np.__version__} | matplotlib: {plt.matplotlib.__version__} | sklearn: {sklearn.__version__} | scipy: {scipy.__version__}")


## 1) Cargar el conjunto de datos (Iris)

Usaremos únicamente dos variables (`sepal length (cm)`, `sepal width (cm)`) para **visualizar en 2D**.


In [ ]:

def load_iris_two_features() -> tuple[np.ndarray, np.ndarray, list[str]]:
    """
    Carga el dataset Iris y devuelve solo dos columnas para visualización 2D.
    Retorna:
        X (np.ndarray): matriz n×2 con [sepal length, sepal width].
        y (np.ndarray): etiquetas verdaderas (0,1,2) para referencia (no se usan para entrenar).
        feature_names (list[str]): nombres de las dos características.
    """
    iris = datasets.load_iris()
    cols = [0, 1]  # 0=sepal length, 1=sepal width
    X = iris.data[:, cols]
    y = iris.target
    feature_names = [iris.feature_names[i] for i in cols]
    return X, y, feature_names

X, y_true, feature_names = load_iris_two_features()
X[:5], feature_names



## 2) Matriz de distancias y dendrograma con SciPy

1. **Estandarizamos** las variables.  
2. Calculamos **distancias por pares** (`pdist`) y las convertimos a matriz cuadrada (`squareform`).  
3. Construimos la **matriz de enlace** (`linkage`) con el método `ward`.


In [ ]:

def compute_distance_and_linkage(X: np.ndarray, method: str = "ward") -> tuple[np.ndarray, np.ndarray]:
    """
    Calcula la matriz de distancias y la matriz de enlace (linkage).
    - method: 'ward', 'single', 'complete', 'average'.
    """
    scaler = StandardScaler()
    X_std = scaler.fit_transform(X)

    dist_vec = pdist(X_std, metric="euclidean")
    D = squareform(dist_vec)

    Z = linkage(dist_vec, method=method)
    return D, Z

D, Z = compute_distance_and_linkage(X, method="ward")
print("Matriz de distancias (primeras 5×5 celdas):\n", np.round(D[:5, :5], 3))



### Dendrograma

> **Tip didáctico:** Identifica una **gran brecha vertical** para elegir un corte razonable (número de clusters).


In [ ]:

def plot_dendrogram(Z: np.ndarray, truncate_mode: str | None = None, p: int = 12) -> None:
    """Dibuja el dendrograma. Usa `truncate_mode` para datasets grandes."""
    plt.figure(figsize=(9, 5))
    dendrogram(
        Z,
        truncate_mode=truncate_mode,  # None = completo
        p=p,
        leaf_rotation=90.0,
        leaf_font_size=10.0,
        show_contracted=False,
    )
    plt.title("Dendrograma (linkage='ward')")
    plt.xlabel("Índice de muestra o (cluster)")
    plt.ylabel("Distancia")
    plt.tight_layout()
    plt.show()

plot_dendrogram(Z, truncate_mode=None)



## 3) Elegir el número de clusters a partir del dendrograma

Dos caminos:

- **Por número de clusters** (`maxclust=k`).  
- **Por umbral de distancia** (`distance_threshold=t`).

A modo de ejemplo con Iris, usaremos **k = 3**.


In [ ]:

def choose_clusters_from_dendrogram(Z: np.ndarray, maxclust: int | None = 3, distance_threshold: float | None = None) -> np.ndarray:
    """Devuelve etiquetas de cluster (1..k) usando `fcluster`."""
    if distance_threshold is not None and maxclust is not None:
        raise ValueError("Usa solo una de las dos opciones: maxclust o distance_threshold.")
    if distance_threshold is not None:
        labels = fcluster(Z, t=distance_threshold, criterion="distance")
    else:
        labels = fcluster(Z, t=maxclust, criterion="maxclust")
    return labels

labels_scipy = choose_clusters_from_dendrogram(Z, maxclust=3, distance_threshold=None)
np.unique(labels_scipy, return_counts=True)



## 4) Clustering con `AgglomerativeClustering` (scikit-learn)

Usamos `linkage='ward'` y **k=3** clusters.


In [ ]:

def sklearn_agglomerative(X: np.ndarray, n_clusters: int = 3, linkage_method: str = "ward") -> np.ndarray:
    """Aplica AgglomerativeClustering y devuelve etiquetas."""
    scaler = StandardScaler()
    X_std = scaler.fit_transform(X)
    model = AgglomerativeClustering(n_clusters=n_clusters, linkage=linkage_method)
    labels = model.fit_predict(X_std)
    return labels

labels_sklearn = sklearn_agglomerative(X, n_clusters=3, linkage_method="ward")
np.unique(labels_sklearn, return_counts=True)



## 5) Visualización 2D de clusters


In [ ]:

def plot_clusters_2d(X: np.ndarray, labels: np.ndarray, feature_names: list[str], title: str) -> None:
    """Grafica puntos en 2D coloreados por etiquetas de cluster."""
    plt.figure(figsize=(6, 5))
    plt.scatter(X[:, 0], X[:, 1], c=labels, s=50, alpha=0.8, edgecolor="k")
    plt.xlabel(feature_names[0])
    plt.ylabel(feature_names[1])
    plt.title(title)
    plt.tight_layout()
    plt.show()

plot_clusters_2d(X, labels_scipy, feature_names, title="Clusters (SciPy fcluster, k=3)")
plot_clusters_2d(X, labels_sklearn, feature_names, title="Clusters (sklearn Agglomerative, k=3)")



### (Opcional) Medida simple de concordancia

**Advertencia didáctica:** Las etiquetas de clustering son **arbitrarias** (permutables). Un acuerdo **etiqueta-a-etiqueta** puede ser bajo aunque los grupos sean equivalentes tras una **re-etiquetación**.


In [ ]:

agreement = np.mean(labels_scipy == labels_sklearn)
print(f"Acuerdo crudo etiqueta-a-etiqueta entre SciPy y sklearn: {agreement:.3f}")  # solo ilustrativo



## ✅ Siguientes pasos y extensiones
- Probar **otros linkage**: `single`, `complete`, `average` y comparar dendrogramas.
- Incorporar **más variables** (p. ej., incluir pétalo) y usar **PCA** para visualizar en 2D.
- Evaluar particiones con **índice de silueta**, **Davies–Bouldin**, etc.
- Explorar el enfoque **divisivo** (implementaciones específicas o bibliotecas complementarias).
- Probar con datasets más grandes y discutir **complejidad** (O(n²) en memoria/tiempo).
